# Step 2: Setup Federated Credentials

![Azure](https://img.shields.io/badge/Azure-0078D4?logo=microsoftazure&logoColor=white)
![OIDC](https://img.shields.io/badge/OIDC-OpenID_Connect-orange?logo=openid&logoColor=white)

This notebook creates federated credentials that link your Azure identities to GitHub repositories, enabling passwordless OIDC authentication.

**What this notebook does:**
1. Loads configuration from `config.json` (or collects it fresh)
2. Builds the repository list (manual or auto-discovered from GitHub)
3. Creates federated credentials for each repo × environment combination
4. Displays a summary of created / skipped / failed credentials

> **Prerequisite:** Run **[01_create_identities.ipynb](01_create_identities.ipynb)** first, or have your Azure identities already created.

## Load Configuration

In [ ]:
import subprocess, json, sys, os

def run_cmd(cmd, capture=True, check=True):
    """Run a shell command and return the result."""
    result = subprocess.run(cmd, shell=True, capture_output=capture, text=True)
    if check and result.returncode != 0:
        print(f"ERROR: {result.stderr.strip()}")
        return None
    return result

# Try to load config from previous notebook
config_path = os.path.join(os.path.dirname(os.path.abspath("__file__")), "config.json")
config = {}
if os.path.exists(config_path):
    with open(config_path) as f:
        config = json.load(f)
    print(f"✓ Loaded config from {config_path}")
    print(json.dumps(config, indent=2))
else:
    print("No config.json found. You'll configure values in the next cell.")

In [ ]:
# --- Extract or collect configuration ---

ORG_NAME = config.get("org_name") or input("GitHub organization name: ").strip()
ENVIRONMENTS = config.get("environments", ["dev", "staging", "production"])
IDENTITY_TYPE = config.get("identity_type") or input("Identity type ('sp' or 'mi'): ").strip().lower()
IDENTITIES = config.get("identities", {})
MI_RESOURCE_GROUP = config.get("mi_resource_group", "")

# If no identities from config, collect them manually
if not IDENTITIES:
    print("\nNo identities found in config. Enter them manually:")
    for env in ENVIRONMENTS:
        if IDENTITY_TYPE == "sp":
            obj_id = input(f"  SP Object ID for '{env}': ").strip()
            IDENTITIES[env] = {"object_id": obj_id}
        else:
            name = input(f"  Managed Identity name for '{env}' (e.g., github-actions-{env}): ").strip()
            IDENTITIES[env] = {"name": name}
    if IDENTITY_TYPE == "mi" and not MI_RESOURCE_GROUP:
        MI_RESOURCE_GROUP = input("  MI resource group: ").strip()

print(f"\nOrganization: {ORG_NAME}")
print(f"Identity type: {'Service Principal' if IDENTITY_TYPE == 'sp' else 'Managed Identity'}")
print(f"Environments: {ENVIRONMENTS}")

## Build Repository List

Choose manual entry or auto-discovery from your GitHub organization.

In [ ]:
# --- Repository List ---
# Choose 'dynamic' to auto-discover or 'manual' to enter names

mode = input("Repo discovery mode ('dynamic' or 'manual'): ").strip().lower()

if mode == "dynamic":
    # Check GitHub CLI
    r = run_cmd("gh auth status", check=False)
    if r and r.returncode != 0:
        print("✗ GitHub CLI not authenticated. Run 'gh auth login' first.")
        sys.exit(1)

    print(f"Fetching repos from {ORG_NAME}...")
    r = run_cmd(f"gh repo list {ORG_NAME} --limit 1000 --json name --jq '.[].name'", check=False)
    if r and r.returncode == 0:
        REPOS = [line.strip() for line in r.stdout.strip().split("\n") if line.strip()]
    else:
        print(f"✗ Failed to fetch repos: {r.stderr.strip() if r else ''}")
        REPOS = []
else:
    repos_input = input("Enter repo names (comma-separated): ").strip()
    REPOS = [r.strip() for r in repos_input.split(",") if r.strip()]

print(f"\n✓ {len(REPOS)} repositories:")
for repo in REPOS:
    print(f"  - {repo}")

## Create Federated Credentials

This creates an OIDC federated credential for each **repo × environment** combination.

Each credential maps:
- **Issuer:** `https://token.actions.githubusercontent.com`
- **Subject:** `repo:<org>/<repo>:environment:<env>`
- **Audience:** `api://AzureADTokenExchange`

In [ ]:
# --- Create Federated Credentials ---

ISSUER = "https://token.actions.githubusercontent.com"
AUDIENCE = "api://AzureADTokenExchange"

total = 0
created = 0
skipped = 0
failed = 0
results = []

for repo in REPOS:
    for env in ENVIRONMENTS:
        total += 1
        subject = f"repo:{ORG_NAME}/{repo}:environment:{env}"
        display_name = f"{repo}-{env}"

        print(f"Creating: {display_name}")

        if IDENTITY_TYPE == "mi":
            mi_name = IDENTITIES.get(env, {}).get("name", f"github-actions-{env}")
            cmd = (
                f'az identity federated-credential create '
                f'--identity-name "{mi_name}" '
                f'--resource-group "{MI_RESOURCE_GROUP}" '
                f'--name "{display_name}" '
                f'--issuer "{ISSUER}" '
                f'--subject "{subject}" '
                f'--audiences "{AUDIENCE}" 2>&1'
            )
        else:
            obj_id = IDENTITIES.get(env, {}).get("object_id", "")
            params = json.dumps({
                "name": display_name,
                "issuer": ISSUER,
                "subject": subject,
                "audiences": [AUDIENCE],
                "description": f"GitHub Actions for {repo} {env} environment"
            })
            cmd = f"az ad app federated-credential create --id \"{obj_id}\" --parameters '{params}' 2>&1"

        r = run_cmd(cmd, check=False)
        output = (r.stdout + r.stderr) if r else ""

        if r and r.returncode == 0:
            created += 1
            status = "✓ Created"
            print(f"  ✓ Created: {display_name}")
        elif "already exists" in output.lower():
            skipped += 1
            status = "⊘ Skipped"
            print(f"  ⊘ Skipped (already exists): {display_name}")
        else:
            failed += 1
            status = "✗ Failed"
            print(f"  ✗ Failed: {display_name}")
            print(f"    {output.strip()[:200]}")

        results.append({"repo": repo, "env": env, "name": display_name, "status": status})

print(f"\n{'='*40}")
print(f"Summary:")
print(f"  Total attempted: {total}")
print(f"  Created:         {created}")
print(f"  Skipped:         {skipped}")
print(f"  Failed:          {failed}")
print(f"{'='*40}")

## Results Table

In [ ]:
# --- Display results as a table ---

header = f"{'Repo':<30} {'Environment':<15} {'Status':<15}"
print(header)
print("-" * len(header))
for r in results:
    print(f"{r['repo']:<30} {r['env']:<15} {r['status']:<15}")

## Save Updated Config

In [ ]:
# --- Update config.json with repo list ---

config["repos"] = REPOS
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

print(f"✓ Config updated with {len(REPOS)} repos.")

## Next Steps

Proceed to **[03_create_environments.ipynb](03_create_environments.ipynb)** to create GitHub environments (dev, staging, production) in each repository.